# Milestone 2 — Parsing & Standardization trên Colab CPU

Xử lý đủ phần M1 của từng thành viên, giữ phiên bản rỗng, đo wall-time/RSS và xuất SQLite cùng ba file Parquet để nhóm dùng cho M3. Theo `context/parsing_new.md`, M2 nộp **repository code và báo cáo Jupyter Book theo nhóm**, không yêu cầu từng thành viên nộp dataset Hugging Face.

Notebook clone repository GitHub và checkout đúng commit đã chốt. Trưởng nhóm commit/push đầy đủ code đã kiểm tra, lấy SHA bằng `git rev-parse HEAD`, rồi điền `COMMIT_SHA` bên dưới trước khi gửi notebook. Để `CODE_MODE='commit'`; không cần đưa ZIP code lên Drive. Notebook kiểm cả commit checkout và fingerprint package, không tự chạy theo nhánh main đang thay đổi.

`COMMIT_SHA` hiện để trống vì bản sửa chưa được người dùng commit/push. Nếu code package thay đổi sau bản đã kiểm tra, cập nhật fingerprint và kiểm chứng lại trước khi bàn giao. Chế độ bundle chỉ giữ làm tùy chọn phục hồi checkpoint cũ.

Thêm shortcut thư mục [arXiv-data](https://drive.google.com/drive/folders/1w9cVHms6Q9aakHKMSJY5DGzmBRAPleeT) vào My Drive. Input nằm ở `Full-data/m1-colab-{OWNER}-01.zip`; output nằm ở `milestone 2/{OWNER}/{RUN_ID}/`. Runtime chọn **CPU**, Python 3.11+. Chạy từng cell lần đầu để kiểm tra đường dẫn trước khi chạy dài.


In [ ]:
from pathlib import Path
import os, sys, re, json, hashlib, shutil, subprocess, zipfile

OWNER = 'trong'  # hoang | huynh-nhi | khang | man-nhi | trong
RUN_MODE = 'fresh'  # fresh hoặc resume
RUN_ID = 'fresh-01'  # mỗi invocation dùng một tên mới, ví dụ resume-02
RESUME_FROM = ''  # đường dẫn Drive: snapshots/snapshot-.../checkpoint.sqlite đã có complete.json
EXPECTED_PUBLICATIONS = 5000
BACKUP_EVERY = 1000

TEAM_FOLDER_ID = '1w9cVHms6Q9aakHKMSJY5DGzmBRAPleeT'
TEAM_DRIVE_DIR = ''  # có thể chỉ định đường dẫn thư mục arXiv-data trong Drive mount
M1_ARCHIVE = ''  # tùy chọn đường dẫn ZIP M1 chính xác
EXPECTED_M1_SHA256 = ''  # nếu nhóm có checksum input độc lập, điền vào đây
CODE_MODE = 'commit'  # clone GitHub theo COMMIT_SHA cố định
CODE_ARCHIVE = ''  # chỉ dùng khi chọn bundle để phục hồi gói code cũ
COMMIT_SHA = ''  
EXPECTED_SOURCE_SHA256 = 'fac05538b049ebc108981025320f539a0e7608f5899eaf037ac8971f00016e67'

assert OWNER in {'hoang','huynh-nhi','khang','man-nhi','trong'}
assert RUN_MODE in {'fresh','resume'}
assert re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_-]*', RUN_ID)
assert (RUN_MODE == 'resume') == bool(RESUME_FROM), 'Resume cần đường dẫn checkpoint; fresh để trống.'
assert sys.version_info >= (3,11)
DRIVE_MOUNT = Path('/content/drive')
LOCAL_WORK_DIR = Path('/content') / f'm2-{OWNER}-{RUN_ID}'
LOCAL_CORPUS_DIR = LOCAL_WORK_DIR / 'corpus'
LOCAL_OUTPUT_DIR = LOCAL_WORK_DIR / 'output'
REPO_DIR = LOCAL_WORK_DIR / 'code'
print(OWNER, RUN_MODE, RUN_ID)


## 1. Mount Drive và cài dependencies

Trong Colab web, cell dưới mở xác thực. Khi dùng CLI, **chạy trước trong terminal**: `colab drivemount -s TEN_PHIEN`. Không gửi token/mã xác thực vào chat. Cell sẽ dùng lại mount đang có.

In [ ]:
if not os.path.ismount(DRIVE_MOUNT):
    from google.colab import drive
    drive.mount(str(DRIVE_MOUNT))
assert os.path.ismount(DRIVE_MOUNT) and (DRIVE_MOUNT/'MyDrive').is_dir()
assert not Path('/dev/nvidia0').exists(), 'Chọn runtime CPU-only.'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                'psutil==5.9.5', 'pyarrow==23.0.1'], check=True)

## 2. Clone đúng commit đã chốt

Trưởng nhóm điền COMMIT_SHA của code đã push, giữ nguyên EXPECTED_SOURCE_SHA256 của bản đã kiểm tra. Notebook clone repository và checkout detached đúng commit đó, xác nhận HEAD và fingerprint package trước khi xử lý. Không cần ZIP code trên Drive.

Mỗi lượt tạo thư mục code mới. Nếu thư mục đã tồn tại, giữ dữ liệu cũ và chọn RUN_ID mới. Restart kernel khi đổi code để tránh module cũ còn trong bộ nhớ.


In [ ]:
def file_sha256(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream,'sha256').hexdigest()

if CODE_MODE == 'commit':
    assert re.fullmatch(r'[0-9a-fA-F]{40}', COMMIT_SHA), 'Trưởng nhóm cần điền full commit SHA đã push; không dùng main/HEAD.'
elif CODE_MODE != 'bundle':
    raise ValueError('CODE_MODE phải là commit hoặc bundle')

# Bootstrap chỉ tìm thư mục nhóm đã có, không tạo fallback cá nhân.
if TEAM_DRIVE_DIR:
    TEAM_PATH = Path(TEAM_DRIVE_DIR).resolve()
    assert TEAM_PATH.is_relative_to(DRIVE_MOUNT.resolve()) and TEAM_PATH.is_dir()
else:
    shortcut = DRIVE_MOUNT/'.shortcut-targets-by-id'/TEAM_FOLDER_ID
    candidates = [shortcut/'arXiv-data', shortcut, DRIVE_MOUNT/'MyDrive/arXiv-data']
    TEAM_PATH = next((p for p in candidates if p.is_dir()), None)
    if TEAM_PATH == shortcut and not (shortcut/'Full-data').is_dir():
        children = [p for p in shortcut.iterdir() if p.is_dir()]
        if len(children) == 1:
            TEAM_PATH = children[0]
    if TEAM_PATH is None:
        raise FileNotFoundError('Thêm shortcut: https://drive.google.com/drive/folders/'+TEAM_FOLDER_ID)

if any(n == 'arxiv_pipeline' or n.startswith('arxiv_pipeline.') for n in sys.modules):
    raise RuntimeError('Restart kernel trước khi nạp lại code; input/output cũ được giữ nguyên.')
if REPO_DIR.exists():
    raise FileExistsError('Code directory đã có; chọn RUN_ID mới hoặc kiểm tra lượt cũ.')
REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
if CODE_MODE == 'bundle':
    bundle = Path(CODE_ARCHIVE) if CODE_ARCHIVE else TEAM_PATH/'milestone 2/code'/f'm2-code-{EXPECTED_SOURCE_SHA256}.zip'
    checksum = bundle.with_suffix(bundle.suffix+'.sha256')
    assert checksum.is_file(), f'Thiếu checksum: {checksum}'
    assert file_sha256(bundle) == checksum.read_text().split()[0], 'Gói code không khớp checksum.'
    REPO_DIR.mkdir()
    with zipfile.ZipFile(bundle) as archive:
        for item in archive.infolist():
            target = (REPO_DIR/item.filename).resolve()
            if not target.is_relative_to(REPO_DIR.resolve()) or '\\' in item.filename:
                raise ValueError('Unsafe code ZIP path')
        archive.extractall(REPO_DIR)
elif CODE_MODE == 'commit':
    subprocess.run(['git','clone','--no-checkout','https://github.com/ChickenDev-cyber/arxiv-data-science.git',str(REPO_DIR)],check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'checkout','--detach',COMMIT_SHA],check=True)
    RESOLVED_COMMIT = subprocess.run(['git','-C',str(REPO_DIR),'rev-parse','HEAD'],
                                     check=True,capture_output=True,text=True).stdout.strip()
    assert RESOLVED_COMMIT == COMMIT_SHA.lower(), 'Checkout khác commit đã chốt.'
else:
    raise ValueError('CODE_MODE phải là bundle hoặc commit')
sys.path.insert(0,str(REPO_DIR/'src'))
from arxiv_pipeline.parsing.benchmark import source_fingerprint
assert source_fingerprint() == EXPECTED_SOURCE_SHA256, 'Fingerprint code khác bản đã chốt.'
from arxiv_pipeline.parsing.colab import (sha256, copy_verified, resolve_team_root,
    stage_corpus, prepare_run, run_prepared, finalize_run)
TEAM_PATH = resolve_team_root(DRIVE_MOUNT, TEAM_FOLDER_ID, str(TEAM_PATH))
DRIVE_OUTPUT_DIR = TEAM_PATH/'milestone 2'/OWNER/RUN_ID
print('Code:', source_fingerprint(), '\nTeam:', TEAM_PATH, '\nOutput:', DRIVE_OUTPUT_DIR)


## 3. Stage input lên ổ local, ngoài timer

Giữ toàn bộ publication và version, bao gồm thư mục rỗng. Archive được hash và danh sách ID/version được lưu. Khi resume phải dùng đúng owner, code, Python và checksum archive của lượt trước. Nếu stage bị gián đoạn, giữ thư mục đó để xem lỗi và chọn RUN_ID mới.

In [ ]:
zip_name = f'm1-colab-{OWNER}-01.zip'
archive_path = Path(M1_ARCHIVE) if M1_ARCHIVE else TEAM_PATH/'Full-data'/zip_name
assert archive_path.is_file(), f'Không thấy {archive_path}; điền M1_ARCHIVE nếu đường dẫn khác.'
if LOCAL_CORPUS_DIR.exists():
    raise FileExistsError('Corpus đã có; không giải nén chồng dữ liệu. Chọn RUN_ID mới.')
LOCAL_WORK_DIR.mkdir(parents=True,exist_ok=True)
local_archive = LOCAL_WORK_DIR/zip_name
assert not local_archive.exists(), 'Archive local đã có; chọn RUN_ID mới để stage sạch.'
input_sha = copy_verified(archive_path,local_archive)
if EXPECTED_M1_SHA256:
    assert input_sha == EXPECTED_M1_SHA256, 'Input khác checksum đã chốt.'
if shutil.disk_usage(LOCAL_WORK_DIR).free < 8*1024**3:
    raise RuntimeError('Cần ít nhất 8 GiB trống trước giải nén; kiểm tra dung lượng.')
INPUT_RECEIPT = stage_corpus(local_archive,LOCAL_CORPUS_DIR,OWNER,
                             expected_count=EXPECTED_PUBLICATIONS)
print({k:v for k,v in INPUT_RECEIPT.items() if k!='arxiv_ids'})

## 4. Fresh hoặc resume, đo cùng lúc xử lý

Parser backup SQLite trên ổ local. Notebook theo dõi backup, chép từng bản vào `snapshots/snapshot-.../` riêng trên Drive, tải lại về local để kiểm SHA-256, SQLite integrity và số bài/checkpoint, rồi mới ghi `complete.json`. Không mở SQLite trên Drive, không ghi đè snapshot cũ. Nếu VM mất, chỉ dùng generation có `complete.json` và checksum hợp lệ; snapshot local có thể mới hơn bản Drive.

Resume: chọn RUN_ID mới và điền RESUME_FROM tới `snapshots/snapshot-.../checkpoint.sqlite` hoàn tất. Notebook tải snapshot về local trước khi restore. Giữ đúng commit/code/Python/input gốc. Checkpoint của code cũ phải dùng đúng commit hoặc bundle cũ; không dùng bản mới để ép bỏ qua các bài từ bản cũ.

Cell chạy sẽ tự lưu kết quả cuối ngay sau khi benchmark kết thúc, trước khi kiểm định. Nếu benchmark/kiểm định lỗi, các file hiện có vẫn được chuyển vào một generation `artifacts/result-.../`, có `artifacts.json` ghi file thành công/lỗi; không ghi `handoff.json` thành công. Giữ nguyên runtime và xem lỗi, có thể chạy lại riêng cell đồng bộ bên dưới.

Wall-time gồm parser imports, processing, backup local, đóng SQLite và xuất Parquet. Staging/install, restore và transfer sau lượt nằm ngoài timer. Guardian truyền Drive chạy ngoài cây tiến trình benchmark nên không nằm trong RSS; có thể cạnh tranh CPU/IO trong lúc parser chạy. Không gọi wall trừ backup là thời gian xử lý thuần. `stdout.log`/`stderr.log` giữ tiến độ. Không sửa code hoặc input khi chạy.


In [ ]:
assert os.path.ismount(DRIVE_MOUNT), 'Drive mount đã mất.'
assert source_fingerprint() == EXPECTED_SOURCE_SHA256
RUN = prepare_run(LOCAL_CORPUS_DIR, LOCAL_OUTPUT_DIR, DRIVE_OUTPUT_DIR,
    owner=OWNER,run_id=RUN_ID,source_sha=EXPECTED_SOURCE_SHA256,
    archive_sha=INPUT_RECEIPT['archive_sha256'],
    resume_from=Path(RESUME_FROM) if RUN_MODE=='resume' else None,backup_every=BACKUP_EVERY,drive_mount=DRIVE_MOUNT)
copy_verified(LOCAL_WORK_DIR/'input.json',DRIVE_OUTPUT_DIR/'input.json')
# Lưu commit thực chạy cùng run identity; các snapshot mang theo run.json này.
RUN['record'].update({'code_mode': CODE_MODE,
    'repository_url': 'https://github.com/ChickenDev-cyber/arxiv-data-science.git' if CODE_MODE=='commit' else None,
    'commit_sha': RESOLVED_COMMIT if CODE_MODE=='commit' else None})
local_record = LOCAL_WORK_DIR/'run.json'
local_record.write_text(json.dumps(RUN['record'],indent=2)+'\n')
copy_verified(local_record,DRIVE_OUTPUT_DIR/'run.json')
# Gói ZIP chỉ dùng cho tùy chọn phục hồi bundle cũ.

if CODE_MODE == 'bundle':
    copy_verified(bundle,DRIVE_OUTPUT_DIR/'code.zip')
print('Benchmark:', RUN['command'])
env = dict(os.environ,PYTHONPATH=str(REPO_DIR/'src'),ARXIV_SENTENCE_BACKEND='rule')
RUN_STATE = run_prepared(RUN, env=env)
# Lưu dữ liệu ngay trong cùng cell, trước mọi assert/kiểm định hậu kỳ.
HANDOFF = finalize_run(LOCAL_CORPUS_DIR, LOCAL_OUTPUT_DIR, DRIVE_OUTPUT_DIR,
                       INPUT_RECEIPT, EXPECTED_SOURCE_SHA256)
print('Handoff complete:', DRIVE_OUTPUT_DIR/'handoff.json')


## 5. Xem kết quả kiểm định

Cell chạy phía trên đã lưu kết quả và kiểm coverage, integrity, tên publication, phiên bản khai báo/parse/rỗng, cạnh tự trỏ/mồ côi/liên bài và số hàng ba Parquet. Thành công vận hành không phải điểm chính xác ngữ nghĩa; custom TeX macros và input M1 thiếu vẫn cần audit.


In [ ]:
VERIFICATION = HANDOFF['verification']
bench = json.loads((LOCAL_OUTPUT_DIR/'benchmark.json').read_text())
print(VERIFICATION)
print('Mode:',bench['run_kind'],'\nWall (s):',bench['wall_seconds'],
      '\nSeconds/new publication:',bench['mean_seconds_per_processed_publication'],
      '\nPeak RSS MiB:',bench['peak_rss_bytes']/1024**2,
      '\nOutput bytes:',bench['artifact_bytes'],'\nStatistics:',bench['statistics'])
print('Giới hạn phép đo:', HANDOFF['measurement_limits'])


## 6. Xác nhận lưu / thử lại sau lỗi

Chỉ có `handoff.json` khi dữ liệu, coverage và snapshot cuối đã qua kiểm tra. File này trỏ tới `artifacts/result-.../` chứa SQLite, ba Parquet, benchmark/pipeline JSON, statistics và logs. `artifacts.json` chỉ là biên nhận truyền dữ liệu, không phải xác nhận parse thành công.

Nếu cell chạy báo lỗi, giữ runtime và chạy cell dưới để thử lưu/kiểm định lại. Mỗi lần thử tạo generation artifacts mới; không xóa bản đã lưu. Không tiếp tục bàn giao nếu chưa có handoff thành công.


In [ ]:
assert os.path.ismount(DRIVE_MOUNT)
if not (DRIVE_OUTPUT_DIR/'handoff.json').is_file():
    HANDOFF = finalize_run(LOCAL_CORPUS_DIR, LOCAL_OUTPUT_DIR, DRIVE_OUTPUT_DIR,
                           INPUT_RECEIPT, EXPECTED_SOURCE_SHA256)
else:
    HANDOFF = json.loads((DRIVE_OUTPUT_DIR/'handoff.json').read_text())
print('Kết quả:', HANDOFF['directory'])
print('Biên nhận:', DRIVE_OUTPUT_DIR/'handoff.json')
